In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# Quiz
- (1) 데이터 수집: 네이버 Open API로 '경주여행', '전주여행' 블로그글 각 500건 수집 (naver.csv)
    * 파일 내용 : query, no, title, link, description, total_text(title + ' ' + description)
- (2) 품사 태깅: 수집한 텍스트(total_text)를 형태소 분석하여 토큰화 (naver_pos.csv)
    * 파일 내용 : query, no, token, pos
- (3) 명사 추출: 분석에 사용할 명사 토큰만 정제하여 추출 (naver_pos_nouns.csv)
    * query, token, pos
- (4) 빈도 분석: 키워드별 지역(경주/전주) 빈도 및 합계 집계 (naver_pos_nouns_count.csv)
    * token, 경주빈도, 전주빈도, 빈도합
- (5) 빈도 시각화: 워드클라우드 및 Text Plot 생성
    * 이미지 저장
- (6) 거리 분석: Word2Vec 모델을 활용한 단어 간 유사도 및 연관성 분석

## 1. 데이터 수집 (네이버 open API)
- query, no, title, link, description, total_text(title + ' ' + description)

In [4]:
# .env가져오기
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='C:/ai/source/01_python/.env')

True

In [12]:
# 네이버 개발자 센터에 있는 소스를 가져오기
# 네이버 검색 API 예제 - 블로그 검색
import os
import sys
import urllib.request

client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

encText = urllib.parse.quote("경주 여행")
url = "https://openapi.naver.com/v1/search/blog.json?query=" + encText

request = urllib.request.Request(url)

request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)

response = urllib.request.urlopen(request)
rescode = response.getcode()

if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8')[:100])
else:
    print("Error Code:" + rescode)

{
	"lastBuildDate":"Thu, 03 Sep 2026 17:23:26 +0900",
	"total":2735078,
	"start":1,
	"display":10,
	


In [13]:
import json
from html import unescape
import requests
import pandas as pd
import re

In [58]:
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

encText = "경주 여행"
url = "https://openapi.naver.com/v1/search/blog.json"

headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}



items_list = []
no = 0

for start in range(1, 501, 100):
    params = {'query': encText, 'display': 100, 'start': start, 'sort': 'date'}
    response = requests.get(url, params=params, headers=headers)
    
    items = response.json()['items']
    
    for item in items:
        no += 1
        query = "경주 여행"
        title = re.sub(r'<[^>]+>','',item['title'])
        title = unescape(title)
        title = re.sub(r'[^a-zA-Z0-9가-힣]',' ',title)
        link  = item['link']
        description = re.sub(r'<[^>]+>','',item['description'])
        description = unescape(description)
        description = re.sub(r'[^a-zA-Z0-9가-힣]',' ',description)
        total_text = title + ' ' + description

        items_list.append([query, no, title, link, description, total_text])

In [59]:
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

encText2 = "전주 여행"
url = "https://openapi.naver.com/v1/search/blog.json"

headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}

items_list2 = []
no2 = 0

for start in range(1, 501, 100):
    params = {'query': encText2, 'display': 100, 'start': start, 'sort': 'date'}
    response2 = requests.get(url, params=params, headers=headers)
    
    items2 = response2.json()['items']
    
    for item2 in items2:
        no2 += 1
        query2 = "전주 여행"
        title2 = re.sub('<[^>]+>','',item2['title'])
        title2 = unescape(title2)
        title2 = re.sub(r'[^a-zA-Z0-9가-힣]',' ',title2)
        link2  = item2['link']
        description2 = re.sub('<[^>]+>','',item2['description'])
        description2 = unescape(description2)
        description2 = re.sub(r'[^a-zA-Z0-9가-힣]',' ',description2)
        total_text2 = title2 + ' ' + description2

        items_list2.append([query2, no2, title2, link2, description2, total_text2])

In [60]:
items_list.extend(items_list2)
df = pd.DataFrame(items_list, columns=['query', 'no', 'title', 'link', 'description', 'total_text'])
df.to_csv('data/naver.csv', index=False, encoding='utf-8')

In [ ]:
def get_search_item_return(query, start):
    pass

In [62]:
from mecab import MeCab
mecab = MeCab()

pos_data = [
    [query, no, token, pos]
    for query, no, title, link, description, total_text in items_list
    for token, pos in mecab.pos(total_text)
]

df_pos = pd.DataFrame(pos_data, columns=['query', 'no', 'token', 'pos'])

df_pos.to_csv('data/naver_pos.csv', index=False, encoding='utf-8-sig')